# 2. Ordinal & Label Encoding

This notebook covers:
1. **Ordinal Encoding (`OrdinalEncoder`)**: Mapping categorical input features ($X$) with an inherent mathematical order.
2. Explicitly defining category rank order via the `categories` parameter.
3. **Label Encoding (`LabelEncoder`)**: Transforming 1D target arrays ($y$) into integer class labels.
4. The critical distinction: Why `LabelEncoder` should never be applied to input feature matrices ($X$).

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

# Sample dataset with mixed features and a target column
data = {
    'Age': [23, 31, 45, 29, 38],
    'Education_Level': ['High School', 'Bachelors', 'PhD', 'Masters', 'Bachelors'],
    'Customer_Rating': ['Poor', 'Good', 'Excellent', 'Average', 'Good'],
    'Loan_Approved': ['No', 'Yes', 'Yes', 'No', 'Yes']  # Target column (y)
}

df = pd.DataFrame(data)
print("=== 1. ORIGINAL DATASET ===")
display(df)

=== 1. ORIGINAL DATASET ===


,Age,Education_Level,Customer_Rating,Loan_Approved
0,23,High School,Poor,No
1,31,Bachelors,Good,Yes
2,45,PhD,Excellent,Yes
3,29,Masters,Average,No
4,38,Bachelors,Good,Yes


---
## Part 1: Ordinal Encoding on Input Features ($X$)

Ordinal data has a meaningful rank:
* **Education:** $\text{High School} < \text{Bachelors} < \text{Masters} < \text{PhD}$
* **Rating:** $\text{Poor} < \text{Average} < \text{Good} < \text{Excellent}$

### The Trap:
If you do not pass explicit rank order, `OrdinalEncoder` orders categories alphabetically (`Average` $\to 0$, `Excellent` $\to 1$, `Good` $\to 2$, `Poor` $\to 3$), breaking the real-world sequence.

### The Fix:
Always supply the ordered lists explicitly via the `categories` parameter.

In [12]:
# 1. Define explicit hierarchy for each ordinal column
education_order = ['High School', 'Bachelors', 'Masters', 'PhD']
rating_order = ['Poor', 'Average', 'Good', 'Excellent']

# 2. Make an explicit copy of the features
df_features = df.copy()

# 3. Configure ColumnTransformer for ordinal features
ordinal_ct = ColumnTransformer(
    transformers=[
        ('ordinal_education', OrdinalEncoder(categories=[education_order]), ['Education_Level']),
        ('ordinal_rating', OrdinalEncoder(categories=[rating_order]), ['Customer_Rating'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

# 4. Transform features 
X_transformed = ordinal_ct.fit_transform(df_features)

print("=== 2. TRANSFORMED FEATURE MATRIX (X) ===")
display(X_transformed)

=== 2. TRANSFORMED FEATURE MATRIX (X) ===


,Education_Level,Customer_Rating,Age,Loan_Approved
0,0.0,0.0,23,No
1,1.0,2.0,31,Yes
2,3.0,3.0,45,Yes
3,2.0,1.0,29,No
4,1.0,2.0,38,Yes


---
## Part 2: Label Encoding on the Target ($y$)

- `LabelEncoder` is strictly designed for **1-dimensional target vectors (`y`)**, not multi-column 2D feature matrices (`X`).
- It maps unique target strings (e.g., `'No'`, `'Yes'`) to integer classes (`0`, `1`).

In [13]:
# 1. Isolate the target column
y = df['Loan_Approved'].copy()

# 2. Initialize and fit LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("=== 3. TARGET ENCODING (y) ===")
print("Original Classes:", le.classes_)
print("Encoded Target Array:", y_encoded)

# Show final combined dataframe with encoded features and target
df_final = X_transformed.copy()
df_final['Loan_Approved'] = y_encoded

print("\n=== 4. FINAL ML-READY DATASET ===")
display(df_final)

=== 3. TARGET ENCODING (y) ===
Original Classes: ['No' 'Yes']
Encoded Target Array: [0 1 1 0 1]

=== 4. FINAL ML-READY DATASET ===


,Education_Level,Customer_Rating,Age,Loan_Approved
0,0.0,0.0,23,0
1,1.0,2.0,31,1
2,3.0,3.0,45,1
3,2.0,1.0,29,0
4,1.0,2.0,38,1


---
## Part 3: Quick Summary Table

| Tool | Input Dimension | Primary Use Case | Example |
| :--- | :--- | :--- | :--- |
| **`OrdinalEncoder`** | 2D Array / DataFrame (`X`) | Ordered input features | `Education`, `Customer Tier` |
| **`LabelEncoder`** | 1D Vector (`y`) | Target classification labels | `Churn`, `Loan_Approved` |
| **`OneHotEncoder`** | 2D Array / DataFrame (`X`) | Unordered (nominal) features | `City`, `Color`, `Device` |